# Create the Project Data Files

This notebook downloads our data from the official sources, saves it as 4 files, and downloads them to your computer.

**How to use:** open in Google Colab and click **Runtime → Run all**. It takes about 5–10 minutes. At the end your browser downloads the 4 files. If it asks, click **Allow** for multiple downloads.

**Sources:**
1. NYC TLC Trip Record Data, yellow taxi trips (main dataset), Oct 2024 – Sep 2025, 80,000 trips sampled per month
2. Open-Meteo, hourly NYC weather
3. MTA Central Business District Taxi Zones, the congestion toll area

## 1. Settings (do not change)

In [ ]:
import pandas as pd

START_MONTH, END_MONTH = "2024-10", "2025-09"
N_PER_MONTH = 80_000
SEED = 42

TRIPS_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{}.parquet"
ZONES_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
WEATHER_URL = ("https://archive-api.open-meteo.com/v1/archive"
               "?latitude=40.7128&longitude=-74.0060"
               "&start_date=2024-10-01&end_date=2025-09-30"
               "&hourly=temperature_2m,precipitation,snowfall,wind_speed_10m"
               "&timezone=America%2FNew_York&format=csv")
CBD_URL = "https://data.ny.gov/api/views/yfdc-w5jh/rows.csv?accessType=DOWNLOAD"

# Only columns known before a trip starts (plus drop-off time for the outcome).
# Fare, tip, tolls and total are left out because they leak the answer.
TRIP_COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
             "trip_distance", "RatecodeID", "PULocationID", "DOLocationID"]

## 2. Taxi trips (main dataset)

In [ ]:
samples = []
for month in pd.period_range(START_MONTH, END_MONTH, freq="M"):
    df = pd.read_parquet(TRIPS_URL.format(month.strftime("%Y-%m")), columns=TRIP_COLS)
    df = df[df["tpep_pickup_datetime"].dt.to_period("M") == month]
    samples.append(df.sample(n=min(N_PER_MONTH, len(df)), random_state=SEED))
    print(month, "done")

trips = pd.concat(samples, ignore_index=True)
trips.to_csv("trips_sample.csv.gz", index=False, compression="gzip")
print("trips_sample.csv.gz:", len(trips), "rows")

## 3. Taxi zone lookup, weather and toll zones

In [ ]:
zones = pd.read_csv(ZONES_URL)
zones.to_csv("taxi_zone_lookup.csv", index=False)
print("taxi_zone_lookup.csv:", len(zones), "rows")

weather = pd.read_csv(WEATHER_URL, skiprows=3)
weather.columns = ["time", "temperature_c", "precipitation_mm", "snowfall_cm", "wind_speed_kmh"]
weather.to_csv("weather_hourly_nyc.csv", index=False)
print("weather_hourly_nyc.csv:", len(weather), "rows")

cbd = pd.read_csv(CBD_URL)
cbd.to_csv("cbd_taxi_zones.csv", index=False)
print("cbd_taxi_zones.csv:", len(cbd), "rows")
print("Toll-zone columns:", list(cbd.columns))

## 4. Download the files to your computer

In [ ]:
from google.colab import files

for name in ["trips_sample.csv.gz", "taxi_zone_lookup.csv",
             "weather_hourly_nyc.csv", "cbd_taxi_zones.csv"]:
    files.download(name)